In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('expense_monitoring_system').getOrCreate()

In [2]:
from google.colab import files
uploaded = files.upload ()

Saving expenses.csv to expenses.csv
Saving users.csv to users.csv


In [3]:
users_df = spark.read.csv("users.csv",header=True,inferSchema=True)

expenses_df = spark.read.csv("expenses.csv",header=True,inferSchema=True)

In [4]:
users_df.show()
users_df.printSchema()

+-------+------------+----------------+----------+--------------+
|user_id|   full_name|           email|      city|monthly_budget|
+-------+------------+----------------+----------+--------------+
|      1|  Arun Kumar|  arun@gmail.com|   Chennai|         25000|
|      2|Priya Sharma| priya@gmail.com|Coimbatore|         30000|
|      3|Rajesh Kumar|rajesh@gmail.com|   Madurai|         20000|
|      4| Anitha Devi|anitha@gmail.com|     Salem|         22000|
|      5|Vikram Singh|vikram@gmail.com|     Erode|         28000|
+-------+------------+----------------+----------+--------------+

root
 |-- user_id: integer (nullable = true)
 |-- full_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- monthly_budget: integer (nullable = true)



In [5]:
expenses_df.show()
expenses_df.printSchema()

+--------------+-------+----------+-------------+------+
|transaction_id|user_id|      date|     category|amount|
+--------------+-------+----------+-------------+------+
|          1001|      1|2026-01-05|         Food|   250|
|          1002|      1|2026-01-06|    Transport|   120|
|          1003|      1|2026-01-08|     Shopping|  1500|
|          1004|      5|2026-01-15|    Utilities|   950|
|          1005|      2|2026-01-05|         Food|   300|
|          1006|      2|2026-01-10|    Utilities|   800|
|          1007|      4|2026-01-20|     Shopping|  2200|
|          1008|      3|2026-01-12|Entertainment|   500|
|          1009|      3|2026-01-13|    Transport|   200|
|          1010|      3|2026-01-18|   Healthcare|  1200|
|          1011|      4|2026-02-02|         Food|   400|
|          1012|      4|2026-02-08|     Shopping|  1800|
|          1013|      3|2026-02-10|Entertainment|   700|
|          1014|      1|2026-02-14|    Transport|   150|
|          1015|      2|2026-02

In [6]:
from pyspark.sql.functions import *

In [7]:
expenses_df = expenses_df.fillna({
    "category": "Unknown",
    "amount": 0.0,
    "date": "2026-03-01"
})

In [8]:
join_df = expenses_df.join(
    users_df,
    "user_id",
    "inner"
)
join_df.show()

+-------+--------------+----------+-------------+------+------------+----------------+----------+--------------+
|user_id|transaction_id|      date|     category|amount|   full_name|           email|      city|monthly_budget|
+-------+--------------+----------+-------------+------+------------+----------------+----------+--------------+
|      1|          1001|2026-01-05|         Food|   250|  Arun Kumar|  arun@gmail.com|   Chennai|         25000|
|      1|          1002|2026-01-06|    Transport|   120|  Arun Kumar|  arun@gmail.com|   Chennai|         25000|
|      1|          1003|2026-01-08|     Shopping|  1500|  Arun Kumar|  arun@gmail.com|   Chennai|         25000|
|      5|          1004|2026-01-15|    Utilities|   950|Vikram Singh|vikram@gmail.com|     Erode|         28000|
|      2|          1005|2026-01-05|         Food|   300|Priya Sharma| priya@gmail.com|Coimbatore|         30000|
|      2|          1006|2026-01-10|    Utilities|   800|Priya Sharma| priya@gmail.com|Coimbatore

In [9]:
users_tot = join_df.groupBy("user_id","full_name").agg(sum("amount").alias("total_spent"))
users_tot.show()

+-------+------------+-----------+
|user_id|   full_name|total_spent|
+-------+------------+-----------+
|      3|Rajesh Kumar|       2600|
|      4| Anitha Devi|       5100|
|      5|Vikram Singh|       1400|
|      2|Priya Sharma|       2750|
|      1|  Arun Kumar|       2320|
+-------+------------+-----------+



In [10]:
users_tot.orderBy(col("total_spent").desc()).show()

+-------+------------+-----------+
|user_id|   full_name|total_spent|
+-------+------------+-----------+
|      4| Anitha Devi|       5100|
|      2|Priya Sharma|       2750|
|      3|Rajesh Kumar|       2600|
|      1|  Arun Kumar|       2320|
|      5|Vikram Singh|       1400|
+-------+------------+-----------+



In [11]:
join_df.orderBy( col("amount").desc()).show(1)

+-------+--------------+----------+--------+------+-----------+----------------+-----+--------------+
|user_id|transaction_id|      date|category|amount|  full_name|           email| city|monthly_budget|
+-------+--------------+----------+--------+------+-----------+----------------+-----+--------------+
|      4|          1007|2026-01-20|Shopping|  2200|Anitha Devi|anitha@gmail.com|Salem|         22000|
+-------+--------------+----------+--------+------+-----------+----------------+-----+--------------+
only showing top 1 row


In [12]:
join_df.groupBy("user_id").max("amount").show()

+-------+-----------+
|user_id|max(amount)|
+-------+-----------+
|      1|       1500|
|      3|       1200|
|      5|        950|
|      4|       2200|
|      2|       1200|
+-------+-----------+



In [13]:
join_df.groupBy("user_id").min("amount").show()

+-------+-----------+
|user_id|min(amount)|
+-------+-----------+
|      1|        120|
|      3|          0|
|      5|        450|
|      4|        400|
|      2|        300|
+-------+-----------+



In [14]:
join_df.groupBy("user_id","category").sum("amount").show()

+-------+-------------+-----------+
|user_id|     category|sum(amount)|
+-------+-------------+-----------+
|      2|     Shopping|       1200|
|      4|    Utilities|        700|
|      3|   Healthcare|       1200|
|      1|         Food|        550|
|      5|    Utilities|        950|
|      3|    Transport|        200|
|      5|      Unknown|        450|
|      4|     Shopping|       4000|
|      2|         Food|        750|
|      2|    Utilities|        800|
|      4|         Food|        400|
|      1|    Transport|        270|
|      3|Entertainment|       1200|
|      1|     Shopping|       1500|
+-------+-------------+-----------+



In [15]:
join_df.groupBy("category").sum("amount").show()

+-------------+-----------+
|     category|sum(amount)|
+-------------+-----------+
|         Food|       1700|
|Entertainment|       1200|
|   Healthcare|       1200|
|      Unknown|        450|
|     Shopping|       6700|
|    Utilities|       2450|
|    Transport|        470|
+-------------+-----------+



In [16]:
print("Report: ")
print("\nTotal users: ",users_df.count())
print("Total expenses: ",expenses_df.count())
print("\nUser with most expenses: ")
users_tot.orderBy(col("total_spent").desc()).show(1)
print("\nUser with least expenses: ")
users_tot.orderBy(col("total_spent").asc()).show(1)
print("\nMost expensive expense: ")
join_df.orderBy( col("amount").desc()).show(1)
print("\nLeast expensive expense: ")
join_df.orderBy( col("amount").asc()).show(1)
print("\nTotal expenses by category: ")
join_df.groupBy("category").sum("amount").show()
print("\nTotal expenses by user: ")
users_tot.show()

Report: 

Total users:  5
Total expenses:  20

User with most expenses: 
+-------+-----------+-----------+
|user_id|  full_name|total_spent|
+-------+-----------+-----------+
|      4|Anitha Devi|       5100|
+-------+-----------+-----------+
only showing top 1 row

User with least expenses: 
+-------+------------+-----------+
|user_id|   full_name|total_spent|
+-------+------------+-----------+
|      5|Vikram Singh|       1400|
+-------+------------+-----------+
only showing top 1 row

Most expensive expense: 
+-------+--------------+----------+--------+------+-----------+----------------+-----+--------------+
|user_id|transaction_id|      date|category|amount|  full_name|           email| city|monthly_budget|
+-------+--------------+----------+--------+------+-----------+----------------+-----+--------------+
|      4|          1007|2026-01-20|Shopping|  2200|Anitha Devi|anitha@gmail.com|Salem|         22000|
+-------+--------------+----------+--------+------+-----------+-----------